In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
jobs = pd.read_csv("../data/processed/jobs_clean.csv")
skills = pd.read_csv("../data/processed/skills_clean.csv")

df = jobs.merge(skills, on="job_id")

In [3]:
transactions = (
    df.groupby("job_id")["skill"]
      .apply(list)
)

transactions.head()

job_id
008JSH07G53Y    [GCP, Computer Vision, R, TensorFlow, AWS]
01IR44A2IUNP        [GCP, Python, R, TensorFlow, SQL, NLP]
01PHCOWT3OML                      [PyTorch, Azure, AWS, R]
01PK13J46SC1                 [PyTorch, R, NLP, TensorFlow]
01SE66E3MOBC                        [GCP, Scikit-learn, R]
Name: skill, dtype: object

In [4]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    te_array,
    columns=te.columns_
)

basket.head()

,AWS,Azure,Computer Vision,GCP,NLP,PyTorch,Python,R,SQL,Scikit-learn,TensorFlow
0,True,False,True,True,False,False,False,True,False,False,True
1,False,False,False,True,True,False,True,True,True,False,True
2,True,True,False,False,False,True,False,True,False,False,False
3,False,False,False,False,True,True,False,True,False,False,True
4,False,False,False,True,False,False,False,True,False,True,False


In [5]:
frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

frequent_itemsets.sort_values(
    "support",
    ascending=False
).head(20)

,support,itemsets
10,0.427317,frozenset({TensorFlow})
3,0.423415,frozenset({GCP})
7,0.418537,frozenset({R})
0,0.416098,frozenset({AWS})
1,0.416098,frozenset({Azure})
6,0.415610,frozenset({Python})
4,0.410732,frozenset({NLP})
8,0.401463,frozenset({SQL})
9,0.400488,frozenset({Scikit-learn})
2,0.398537,frozenset({Computer Vision})


In [6]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

rules = rules.sort_values(
    "lift",
    ascending=False
)

rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [7]:
rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

,antecedents,consequents,support,confidence,lift


In [9]:
rules = association_rules(

    frequent_itemsets,

    metric="confidence",

    min_threshold=0.5

)

# Convert numeric columns

numeric_cols = ["support", "confidence", "lift", "leverage", "conviction"]

for col in numeric_cols:

    rules[col] = pd.to_numeric(rules[col], errors="coerce")

rules = rules.dropna()

In [10]:
top_rules = rules.nlargest(15, "lift")

display(top_rules)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [11]:
pair_itemsets = frequent_itemsets[
    frequent_itemsets["itemsets"].apply(len) == 2
]

pair_itemsets.sort_values(
    "support",
    ascending=False
).head(20)

,support,itemsets
62,0.170732,"frozenset({TensorFlow, R})"
44,0.170732,"frozenset({TensorFlow, GCP})"
50,0.167805,"frozenset({TensorFlow, NLP})"
59,0.167317,"frozenset({TensorFlow, Python})"
13,0.165854,"frozenset({AWS, GCP})"
29,0.165366,"frozenset({TensorFlow, Azure})"
25,0.164878,"frozenset({Azure, Python})"
20,0.163902,"frozenset({AWS, TensorFlow})"
38,0.163415,"frozenset({GCP, NLP})"
16,0.163415,"frozenset({AWS, Python})"


In [12]:
triplets = frequent_itemsets[
    frequent_itemsets["itemsets"].apply(len) == 3
]

triplets.sort_values(
    "support",
    ascending=False
).head(20)

,support,itemsets
89,0.065366,"frozenset({AWS, GCP, TensorFlow})"
171,0.062927,"frozenset({TensorFlow, R, Computer Vision})"
131,0.062927,"frozenset({TensorFlow, Azure, NLP})"
166,0.062927,"frozenset({SQL, Computer Vision, Python})"
119,0.062927,"frozenset({GCP, Azure, NLP})"
86,0.062439,"frozenset({AWS, R, GCP})"
180,0.061951,"frozenset({TensorFlow, GCP, NLP})"
140,0.061463,"frozenset({TensorFlow, Azure, Python})"
185,0.060976,"frozenset({TensorFlow, GCP, PyTorch})"
210,0.060488,"frozenset({TensorFlow, NLP, Scikit-learn})"


In [13]:
rules.to_csv(
    "../data/processed/association_rules.csv",
    index=False
)

In [14]:
print("Top 10 Most Valuable Skill Associations")

display(
    rules[
        [
            "antecedents",
            "consequents",
            "confidence",
            "lift"
        ]
    ].head(10)
)

Top 10 Most Valuable Skill Associations


,antecedents,consequents,confidence,lift
